# Train TrOCR on IAM Handwriting Dataset

This notebook fine-tunes the Microsoft TrOCR model on the IAM Handwriting dataset using Hugging Face datasets (`Teklia/IAM-line`).

### ⚠️ IMPORTANT: Enable GPU ⚠️
Go to **Runtime** > **Change runtime type** > Select **T4 GPU**.

In [ ]:
import torch
if torch.cuda.is_available():
    print(f"✅ GPU Detected: {torch.cuda.get_device_name(0)}")
else:
    print("❌ GPU NOT Detected! Please change runtime type to GPU.")
    # raise RuntimeError("No GPU found. Training will be too slow.")

In [ ]:
# MOUNT GOOGLE DRIVE (MANDATORY)
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive mounted successfully.")

In [ ]:
# 1. Clone or Update the repository
import os

if os.path.exists('handwriting_recog'):
    %cd handwriting_recog
    !git pull origin main
else:
    !git clone https://github.com/Bhuvan-018/handwriting_recog
    %cd handwriting_recog

In [ ]:
# 2. Install dependencies
!pip install -r requirements.txt
!pip install huggingface_hub

In [ ]:
# 3. Run the training script (SKIP THIS since you have the model)
# !python train_hf.py

In [ ]:
# 4. Restore Model from Google Drive and Deploy
import os
import shutil
import glob
from huggingface_hub import HfApi, login

# --- Configuration ---
HF_TOKEN = "YOUR_HF_WRITE_TOKEN" # @param {type:"string"}
SPACE_ID = "bhuvan-018/handwriting-recognition" # @param {type:"string"}

# Clean inputs
HF_TOKEN = HF_TOKEN.strip()
SPACE_ID = SPACE_ID.strip()

DEPLOY_DIR = "deploy_package"
MODEL_EXTRACT_DIR = "models/trocr_finetuned_iam_hf"

# --- SPECIFIC ZIP FILE PATH ---
ZIP_FILE_PATH = "/content/drive/MyDrive/Handwriting_Model_Backup/trocr_model_20260301_1159.zip"

if os.path.exists(ZIP_FILE_PATH):
    print(f"✅ Found model zip: {ZIP_FILE_PATH}")
    
    # --- Unzip Model ---
    print(f"📦 Unzipping {ZIP_FILE_PATH}...")
    !unzip -o "{ZIP_FILE_PATH}" -d .
    
    # Verify extraction path
    if os.path.exists(MODEL_EXTRACT_DIR):
        print(f"✅ Model extracted to: {MODEL_EXTRACT_DIR}")
    else:
        # If unzip structure is different, find where it went
        if os.path.exists("trocr_finetuned_iam_hf"):
             shutil.move("trocr_finetuned_iam_hf", "models/trocr_finetuned_iam_hf")
             print("✅ Model extracted and moved to correct directory.")
        elif os.path.exists("models/trocr_finetuned_iam_hf"):
             print("✅ Model extracted to models/trocr_finetuned_iam_hf")
        else:
             print("⚠️ Could not automatically verify model path. Listing current dir:")
             print(os.listdir("."))
             if os.path.exists("models"):
                 print("Listing models dir:")
                 print(os.listdir("models"))
    
    # --- Deploy ---
    if HF_TOKEN == "YOUR_HF_WRITE_TOKEN" or not HF_TOKEN:
        print("⚠️ Please enter your Hugging Face Write Token above!")
    else:
        try:
            login(token=HF_TOKEN)
            print("✅ Authenticated with Hugging Face.")
            
            # --- Prepare Deployment Package ---
            print(f"📦 Preparing deployment package in '{DEPLOY_DIR}'...")
            if os.path.exists(DEPLOY_DIR):
                shutil.rmtree(DEPLOY_DIR)
            os.makedirs(DEPLOY_DIR)
            
            # 1. Copy App Files
            if os.path.exists("app_gradio.py"):
                shutil.copy("app_gradio.py", f"{DEPLOY_DIR}/app.py")
            else:
                os.system(f"wget https://raw.githubusercontent.com/Bhuvan-018/handwriting_recog/main/app_gradio.py -O {DEPLOY_DIR}/app.py")

            if os.path.exists("requirements.txt"):
                shutil.copy("requirements.txt", f"{DEPLOY_DIR}/requirements.txt")
            else:
                os.system(f"wget https://raw.githubusercontent.com/Bhuvan-018/handwriting_recog/main/requirements.txt -O {DEPLOY_DIR}/requirements.txt")
                
            if os.path.exists("utils"):
                shutil.copytree("utils", f"{DEPLOY_DIR}/utils")
            else:
                os.makedirs(f"{DEPLOY_DIR}/utils", exist_ok=True)
                os.system(f"wget https://raw.githubusercontent.com/Bhuvan-018/handwriting_recog/main/utils/preprocessing.py -O {DEPLOY_DIR}/utils/preprocessing.py")

            # 2. Copy Model Files
            # Remove checkpoints first to save space
            print("🧹 Cleaning up intermediate checkpoints...")
            if os.path.exists(MODEL_EXTRACT_DIR):
                checkpoints = [d for d in os.listdir(MODEL_EXTRACT_DIR) if d.startswith('checkpoint-')]
                for ckpt in checkpoints:
                    shutil.rmtree(os.path.join(MODEL_EXTRACT_DIR, ckpt))
                
                target_model_dir = f"{DEPLOY_DIR}/models/trocr_finetuned_iam_hf"
                shutil.copytree(MODEL_EXTRACT_DIR, target_model_dir)
                print(f"   - Model files copied to {target_model_dir}")
            else:
                print(f"❌ Critical Error: Source model dir {MODEL_EXTRACT_DIR} not found for copying!")

            # 3. Upload
            print("🚀 Uploading to Hugging Face Space (this handles large files automatically)...")
            api = HfApi()
            api.upload_folder(
                folder_path=DEPLOY_DIR,
                repo_id=SPACE_ID,
                repo_type="space",
                commit_message="Deploy restored model from Drive",
                ignore_patterns=[".git", ".ipynb_checkpoints"]
            )
            print("✅ Successfully deployed to Hugging Face Space!")
            print(f"🔗 Check it out here: https://huggingface.co/spaces/{SPACE_ID}")
            
        except Exception as e:
            print(f"❌ Deployment failed: {e}")
else:
    print(f"❌ Error: Zip file not found at {ZIP_FILE_PATH}")
    print("Please verify the path in Google Drive.")